# 06 — Activation Extraction Pipeline

Extract hidden-state activations at **"I" token positions** from Mistral-7B-Instruct-v0.3
responses to 2,000 prompts across 20 categories.

**Goal:** Build the [`ajbr0wn/self-axis-activations`](https://huggingface.co/datasets/ajbr0wn/self-axis-activations)
HuggingFace dataset for downstream geometry analysis (MODEL_BOUND vs OTHER_BOUND).

| Parameter | Value |
|---|---|
| Model | `mistralai/Mistral-7B-Instruct-v0.3` |
| Precision | fp16 (no quantization — clean activations) |
| Layers | All 32 transformer layers |
| Hidden dim | 4096 |
| Compute | Google Colab T4 (16 GB VRAM) |

## 1. Setup

In [ ]:
!pip install -q transformers datasets huggingface_hub accelerate pyyaml safetensors

In [ ]:
import torch
import yaml
import json
import os
import numpy as np
from pathlib import Path
from datetime import datetime
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset, load_dataset, concatenate_datasets
from huggingface_hub import login
from google.colab import drive

# ── Config ──────────────────────────────────────────────────────────────────
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.3"
HF_DATASET = "ajbr0wn/self-axis-activations"
REPO_URL = "https://github.com/ajbr0wn/self-axis.git"
CHECKPOINT_DIR = "/content/drive/MyDrive/self-axis/checkpoints"

MAX_ATTEMPTS = 10          # max generation attempts per prompt
PASS_THRESHOLD = 3         # minimum "I" count to consider a prompt "passed"
MAX_NEW_TOKENS = 256       # max response length
PUSH_EVERY = 5             # push to HuggingFace every N prompts

GENERATION_CONFIG = dict(
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
)

In [ ]:
# ── Mount Google Drive (persistent checkpoint storage) ──────────────────────
drive.mount("/content/drive")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── HuggingFace login ──────────────────────────────────────────────────────
login()  # paste your HF token when prompted

## 2. Load model

fp16, no quantization. We need clean activations for geometry analysis.

In [ ]:
print("Loading model (fp16, no quantization)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
model.eval()

print(f"Model loaded on {model.device}")
mem_used = torch.cuda.memory_allocated() / 1e9
mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU memory: {mem_used:.1f} GB / {mem_total:.1f} GB")

## 3. Discover "I" token IDs

SentencePiece encodes `"I"` differently depending on context:
- **`▁I`** (with leading space marker) — mid-sentence
- **`I`** (bare) — start of sequence or after special tokens

We need both IDs. We also verify they don't collide with tokens inside other words.

In [ ]:
test_text = "I think that I am here"
test_ids = tokenizer.encode(test_text, add_special_tokens=False)
test_tokens = tokenizer.convert_ids_to_tokens(test_ids)

print("Token analysis for: '%s'" % test_text)
for i, (tid, tok) in enumerate(zip(test_ids, test_tokens)):
    decoded = tokenizer.decode([tid])
    print(f"  pos {i}: id={tid:>6d}  token='{tok}'  decoded='{decoded}'")

# Collect IDs for standalone "I"
I_TOKEN_IDS = set()
for tid, tok in zip(test_ids, test_tokens):
    stripped = tok.replace("▁", "").strip()
    if stripped == "I":
        I_TOKEN_IDS.add(tid)

print(f"\n'I' token IDs: {I_TOKEN_IDS}")

# Verify no collision with common I-starting words
print("\nCollision check:")
for word in ["In", "It", "Is", "If", "Ice", "Imagine", "I'm", "I'll", "I've"]:
    word_ids = tokenizer.encode(word, add_special_tokens=False)
    overlap = set(word_ids) & I_TOKEN_IDS
    status = "⚠️  OVERLAP" if overlap else "✓"
    print(f"  {status}  '{word}' → {word_ids}")

## 4. Load prompts

In [ ]:
# Clone the repo (skip if already present)
if not os.path.exists("/content/self-axis"):
    !git clone {REPO_URL} /content/self-axis

with open("/content/self-axis/data/prompts.yaml", "r") as f:
    data = yaml.safe_load(f)

prompts = []
for cat in data["categories"]:
    for idx, prompt_text in enumerate(cat["prompts"]):
        prompts.append({
            "prompt_id": f"{cat['id']}_{idx:03d}",
            "category": cat["id"],
            "group": cat["group"],
            "prompt_text": prompt_text,
        })

print(f"Loaded {len(prompts)} prompts")
print(f"  MODEL_BOUND:  {sum(1 for p in prompts if p['group'] == 'model_bound')}")
print(f"  OTHER_BOUND:  {sum(1 for p in prompts if p['group'] == 'other_bound')}")
print(f"  Categories:   {len(data['categories'])}")

## 5. Checkpoint management

Each response is saved to Google Drive as a `.pt` file immediately after extraction.
On notebook restart, we query **HuggingFace** (via streaming) to determine which
prompts are already done — this supports resuming across different Colab accounts.

Google Drive remains the crash-safety mechanism; HF is the source of truth for progress.

In [ ]:
def get_completed_prompts():
    """Query HuggingFace dataset for completed prompts (streaming to save memory)."""
    try:
        ds = load_dataset("ajbr0wn/self-axis-activations", split="train", streaming=True)
        completed = {}
        count = 0
        for row in ds:
            pid = row["prompt_id"]
            if pid not in completed:
                completed[pid] = {"attempts": [], "passed": False}
            completed[pid]["attempts"].append(row["attempt_number"])
            if row["prompt_passed"]:
                completed[pid]["passed"] = True
            count += 1
        print(f"Found {count} rows from HuggingFace")
        return completed
    except Exception as e:
        print(f"Could not load from HF: {e}")
        print("Starting fresh...")
        return {}


def save_row(row):
    """Save a single response row to Google Drive checkpoint."""
    fname = f"{row['prompt_id']}_attempt{row['attempt_number']}.pt"
    torch.save(row, os.path.join(CHECKPOINT_DIR, fname))


def flag_prompt(prompt_id):
    """Mark all stored rows for this prompt as flagged."""
    for fname in os.listdir(CHECKPOINT_DIR):
        if fname.startswith(prompt_id + "_attempt") and fname.endswith(".pt"):
            fpath = os.path.join(CHECKPOINT_DIR, fname)
            row = torch.load(fpath, map_location="cpu")
            row["prompt_flagged"] = True
            torch.save(row, fpath)


# ── Check existing progress ────────────────────────────────────────────────
completed = get_completed_prompts()
n_passed = sum(1 for v in completed.values() if v["passed"])
n_flagged = sum(
    1 for v in completed.values()
    if not v["passed"] and max(v["attempts"]) >= MAX_ATTEMPTS
)
n_remaining = len(prompts) - len(completed)
print(f"Checkpoint state:")
print(f"  Passed:    {n_passed}")
print(f"  Flagged:   {n_flagged}")
print(f"  Remaining: {n_remaining}")

## 6. Helper functions

**Two-pass approach:**
1. `model.generate()` — produce the response (no hidden states needed)
2. `model(full_sequence, output_hidden_states=True)` — single forward pass to
   extract activations at every layer

This works because causal attention means the hidden state at each position
is identical whether computed autoregressively or in a single pass.

In [ ]:
def generate_response(prompt_text):
    """Generate a response → (full_token_ids, response_text, prompt_length)."""
    messages = [{"role": "user", "content": prompt_text}]
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    input_ids = tokenizer(input_text, return_tensors="pt").input_ids.to(model.device)
    prompt_len = input_ids.shape[1]

    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            pad_token_id=tokenizer.eos_token_id,
            **GENERATION_CONFIG,
        )

    full_ids = output_ids[0]  # [total_seq_len]
    response_text = tokenizer.decode(full_ids[prompt_len:], skip_special_tokens=True)
    return full_ids, response_text, prompt_len


def find_i_positions(token_ids, response_start):
    """Find positions of standalone 'I' tokens in the response portion only."""
    if isinstance(token_ids, torch.Tensor):
        token_ids = token_ids.tolist()
    return [
        pos for pos in range(response_start, len(token_ids))
        if token_ids[pos] in I_TOKEN_IDS
    ]


def extract_activations(full_ids, positions):
    """Forward pass → list of [32, 4096] activation tensors at each position."""
    with torch.no_grad():
        outputs = model(
            full_ids.unsqueeze(0),
            output_hidden_states=True,
            return_dict=True,
        )

    # outputs.hidden_states: tuple of 33 tensors [1, seq_len, 4096]
    # Index 0 = embedding layer, 1–32 = transformer layers
    hidden_states = torch.stack(outputs.hidden_states[1:], dim=0)  # [32, 1, seq, 4096]

    activations = []
    for pos in positions:
        act = hidden_states[:, 0, pos, :].cpu().to(torch.float16)  # [32, 4096]
        activations.append(act)

    del outputs, hidden_states
    torch.cuda.empty_cache()
    return activations


# Reusable zero tensor for missing activation slots
ZERO_ACT = torch.zeros(32, 4096, dtype=torch.float16)


def build_row(prompt_info, response_text, attempt, passed, pass_attempt,
              i_count, i_positions, activations):
    """Assemble a row dict for checkpoint storage."""
    return {
        "prompt_id":          prompt_info["prompt_id"],
        "category":           prompt_info["category"],
        "group":              prompt_info["group"],
        "prompt_text":        prompt_info["prompt_text"],
        "response_text":      response_text,
        "attempt_number":     attempt,
        "prompt_passed":      passed,
        "prompt_pass_attempt": pass_attempt if pass_attempt is not None else -1,
        "prompt_flagged":     False,
        "i_count":            i_count,
        "i_positions":        i_positions,
        "i_1_activations":    activations[0] if len(activations) > 0 else ZERO_ACT,
        "i_2_activations":    activations[1] if len(activations) > 1 else ZERO_ACT,
        "i_3_activations":    activations[2] if len(activations) > 2 else ZERO_ACT,
        "i_extra_activations": activations[3:] if len(activations) > 3 else [],
    }

## 7. Smoke test

Run one prompt end-to-end before committing to the full loop.
Verify shapes, token detection, and memory usage.

In [ ]:
test_prompt = prompts[0]
print(f"Prompt: \"{test_prompt['prompt_text']}\"")
print(f"  id:       {test_prompt['prompt_id']}")
print(f"  category: {test_prompt['category']}")
print(f"  group:    {test_prompt['group']}")
print()

# Generate
full_ids, response_text, prompt_len = generate_response(test_prompt["prompt_text"])
print(f"Response ({len(full_ids) - prompt_len} tokens):")
print(f"  {response_text[:300]}{'...' if len(response_text) > 300 else ''}")
print()

# Find "I" tokens
i_pos = find_i_positions(full_ids, prompt_len)
print(f"Found {len(i_pos)} 'I' token(s) at positions: {i_pos}")

# Show context around each "I"
for pos in i_pos[:5]:  # show first 5
    context_start = max(prompt_len, pos - 3)
    context_end = min(len(full_ids), pos + 4)
    context_ids = full_ids[context_start:context_end].tolist()
    context_tokens = [tokenizer.decode([tid]) for tid in context_ids]
    marker = ["→" if i == pos - context_start else " " for i in range(len(context_tokens))]
    print(f"  pos {pos}: {''.join(f'{m}[{t}]' for m, t in zip(marker, context_tokens))}")
print()

# Extract activations
if i_pos:
    acts = extract_activations(full_ids, i_pos)
    print(f"Activations extracted: {len(acts)} × {acts[0].shape}")
    print(f"  dtype:  {acts[0].dtype}")
    print(f"  stats:  min={acts[0].min():.4f}  max={acts[0].max():.4f}  mean={acts[0].float().mean():.4f}")
    print(f"  bytes:  {acts[0].nbytes:,} per position ({acts[0].nbytes / 1024:.0f} KB)")
    del acts
else:
    print("⚠️  No 'I' tokens found — try running again or check a different prompt")

del full_ids
torch.cuda.empty_cache()

mem_used = torch.cuda.memory_allocated() / 1e9
mem_total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"\nGPU memory after cleanup: {mem_used:.1f} GB / {mem_total:.1f} GB")
print("\n✓ Smoke test passed!")

## 8. Compile & push to HuggingFace

Compiles local `.pt` checkpoint files and **merges** them with any existing data
on HuggingFace. This supports running extraction across multiple accounts or
sessions — new rows are appended without replacing what's already there.

In [ ]:
def compile_and_push():
    """Compile local checkpoints and MERGE with existing HuggingFace dataset."""
    existing_keys = set()
    try:
        ds_stream = load_dataset(HF_DATASET, split="train", streaming=True)
        for row in ds_stream:
            key = f"{row['prompt_id']}_attempt{row['attempt_number']}"
            existing_keys.add(key)
        print(f"Found {len(existing_keys)} existing rows on HF")
    except:
        print("No existing dataset found on HF")

    new_rows = []
    for fname in sorted(os.listdir(CHECKPOINT_DIR)):
        if not fname.endswith(".pt"):
            continue
        key = fname.replace(".pt", "")
        if key in existing_keys:
            continue
        row = torch.load(os.path.join(CHECKPOINT_DIR, fname), map_location="cpu")
        new_rows.append({
            "prompt_id":          row["prompt_id"],
            "category":           row["category"],
            "group":              row["group"],
            "prompt_text":        row["prompt_text"],
            "response_text":      row["response_text"],
            "attempt_number":     row["attempt_number"],
            "prompt_passed":      row["prompt_passed"],
            "prompt_pass_attempt": row["prompt_pass_attempt"],
            "prompt_flagged":     row["prompt_flagged"],
            "i_count":            row["i_count"],
            "i_positions":        row["i_positions"],
            "i_1_activations":    row["i_1_activations"].numpy(),
            "i_2_activations":    row["i_2_activations"].numpy(),
            "i_3_activations":    row["i_3_activations"].numpy(),
        })

    if not new_rows:
        print("No new rows to push.")
        return

    new_ds = Dataset.from_list(new_rows)
    if existing_keys:
        existing_ds = load_dataset(HF_DATASET, split="train")
        combined = concatenate_datasets([existing_ds, new_ds])
    else:
        combined = new_ds

    combined.push_to_hub(HF_DATASET, private=True)
    print(f"✓ Pushed {len(new_rows)} new rows (total: {len(combined)})")

## 9. Main extraction loop

For each prompt:
1. Generate up to 10 responses
2. Store every response with ≥1 "I" token (activations + metadata)
3. Stop early once a response achieves ≥3 "I" tokens ("pass")
4. Flag the prompt if 10 attempts never reach the pass threshold

Checkpoints to Drive after every response. Pushes to HF periodically.

In [ ]:
# ── Determine remaining work ───────────────────────────────────────────────
completed = get_completed_prompts()

remaining = [
    p for p in prompts
    if p["prompt_id"] not in completed
    or (
        not completed[p["prompt_id"]]["passed"]
        and max(completed[p["prompt_id"]]["attempts"]) < MAX_ATTEMPTS
    )
]

print(f"Prompts to process: {len(remaining)}")
print(f"  (already passed: {sum(1 for v in completed.values() if v['passed'])})")
print()

# ── Run ────────────────────────────────────────────────────────────────────
passed_count = 0
flagged_count = 0
total_responses = 0
start_time = datetime.now()

for i, prompt_info in enumerate(remaining):
    pid = prompt_info["prompt_id"]

    # Resume from last attempt if partially done
    start_attempt = 1
    if pid in completed:
        start_attempt = max(completed[pid]["attempts"]) + 1

    attempt = start_attempt
    passed = False
    pass_attempt = None

    while attempt <= MAX_ATTEMPTS:
        try:
            # Generate
            full_ids, response_text, prompt_len = generate_response(
                prompt_info["prompt_text"]
            )

            # Find "I" tokens in response
            i_pos = find_i_positions(full_ids, prompt_len)
            i_count = len(i_pos)

            if i_count >= 1:
                # Extract activations (forward pass)
                activations = extract_activations(full_ids, i_pos)

                if i_count >= PASS_THRESHOLD and not passed:
                    passed = True
                    pass_attempt = attempt

                # Save to Drive
                row = build_row(
                    prompt_info, response_text, attempt,
                    passed, pass_attempt, i_count, i_pos, activations,
                )
                save_row(row)
                total_responses += 1

                del activations

            del full_ids
            torch.cuda.empty_cache()

            if passed:
                break

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                print(f"  ⚠️  OOM on {pid} attempt {attempt} — clearing cache and retrying")
                torch.cuda.empty_cache()
            else:
                raise

        attempt += 1

    if passed:
        passed_count += 1
    else:
        flag_prompt(pid)
        flagged_count += 1

    # ── Progress ───────────────────────────────────────────────────────────
    if (i + 1) % 10 == 0:
        elapsed = (datetime.now() - start_time).total_seconds()
        rate = (i + 1) / elapsed * 60  # prompts per minute
        eta = (len(remaining) - i - 1) / rate if rate > 0 else 0
        print(
            f"[{i+1}/{len(remaining)}]  "
            f"passed={passed_count}  flagged={flagged_count}  "
            f"responses={total_responses}  "
            f"rate={rate:.1f}/min  ETA={eta:.0f}min"
        )

    # ── Periodic HF push ───────────────────────────────────────────────────
    if (i + 1) % PUSH_EVERY == 0:
        print(f"  → Pushing to HuggingFace...")
        try:
            compile_and_push()
        except Exception as e:
            print(f"  → Push failed: {e} (will retry later)")

print()
print(f"═══ Loop complete ═══")
print(f"  Passed:    {passed_count}")
print(f"  Flagged:   {flagged_count}")
print(f"  Responses: {total_responses}")
print(f"  Time:      {(datetime.now() - start_time).total_seconds() / 60:.1f} min")

## 10. Final push to HuggingFace

In [ ]:
compile_and_push()

## 11. Summary statistics

In [ ]:
completed = get_completed_prompts()

n_passed = sum(1 for v in completed.values() if v["passed"])
n_flagged = sum(
    1 for v in completed.values()
    if not v["passed"] and max(v.get("attempts", [0])) >= MAX_ATTEMPTS
)
n_remaining = len(prompts) - len(completed)
n_responses = len([f for f in os.listdir(CHECKPOINT_DIR) if f.endswith(".pt")])

print("═══════════════════════════════════════")
print("  Self-Axis Extraction Summary")
print("═══════════════════════════════════════")
print(f"  Total prompts:   {len(prompts)}")
print(f"  Passed (≥3 I):   {n_passed}")
print(f"  Flagged:         {n_flagged}")
print(f"  Remaining:       {n_remaining}")
print(f"  Total responses: {n_responses}")
print("═══════════════════════════════════════")

# ── Per-category breakdown ─────────────────────────────────────────────────
print("\nPer-category pass rates:")
from collections import Counter
cat_passed = Counter()
cat_total = Counter()
for pid, info in completed.items():
    cat = pid.rsplit("_", 1)[0]
    cat_total[cat] += 1
    if info["passed"]:
        cat_passed[cat] += 1

for cat in sorted(cat_total.keys()):
    p, t = cat_passed[cat], cat_total[cat]
    bar = "█" * (p * 20 // t) + "░" * (20 - p * 20 // t)
    print(f"  {cat:<45s} {bar} {p}/{t}")

# ── Flagged prompts ────────────────────────────────────────────────────────
flagged_pids = [
    pid for pid, info in completed.items()
    if not info["passed"] and max(info.get("attempts", [0])) >= MAX_ATTEMPTS
]
if flagged_pids:
    print(f"\n⚠️  {len(flagged_pids)} flagged prompt(s):")
    for pid in sorted(flagged_pids):
        print(f"    {pid}")
else:
    print("\n✓ No flagged prompts!")